# aioli-bike-sharing

This notebook shows a minimal **MLIS** workflow using `aioli`:

1. Read your internal S3 model URI (from `best-model-uri.txt`)
2. Create an S3-backed **model registry**
3. Create a **packaged model**
4. Create a secured **deployment**
5. Fetch the deployment **endpoint** and send an inference request

The AIOLI CLI can be extended to deploy additional models; the example below demonstrates one such use case. Refer the documentation for more details


**Docs**
- MLIS documentation: https://support.hpe.com/hpesc/public/docDisplay?docId=a00aie18hen_us&docLocale=en_US&page=MLIS%2Fmlis.html
- aioli deployment quickstart: https://support.hpe.com/hpesc/public/docDisplay?docId=a00aie18hen_us&page=MLIS%2Fget-started%2Faioli-dep-quickstart.html




## Set up environment variables

AIOLI-cli reads the environment variables during its execution.


This cell derives:
- `BUCKET_NAME` from the bucket portion of your `best-model-uri.txt`
- `MODEL_URL` (example assumes a `model.pkl` inside that model artifact path)
- `ENDPOINT_URL` for the internal S3 proxy (adjust for your cluster)
- `SERVE_IMAGE` for MLServer (adjust if you use a different runtime/image)


In [ ]:
import os

# path to best model uri 
os.environ["BUCKET_NAME"]=open("best-model-uri.txt").read().strip().split("://", 1)[1].split("/", 1)[0] 
os.environ["MODEL_URL"]=open("best-model-uri.txt").read().strip().rstrip("/")+"/model.pkl"

os.environ["ENDPOINT_URL"]=os.environ["AWS_ENDPOINT_URL"] # Enter the internal or enternal endpoint url
os.environ["SERVE_IMAGE"]="docker.io/seldonio/mlserver:1.5.0"  

os.environ["MODEL_NAME"]="bike-sharing" # Enter the model name
os.environ["DEPLOYMENT_NAME"] = "bike-sharing-dep" # Enter the deployment name
os.environ["PACKAGED_MODEL"] = "bike-sharing-model" # Enter the packaged model name

## 1) Create / verify the model registry

Create an S3-compatible registry that connects to an internal or external object store.

If your environment needs valid S3 credentials for registry validation, configure the following:

S3_ACCESS_KEY
S3_SECRET_KEY

If the platform ignores credentials for the internal S3 proxy, the CLI still requires the flags, but values may be ignored.


In [ ]:
!aioli registries create mlflow \
  --type s3 \
  --bucket "$BUCKET_NAME" \
  --access-key "${S3_ACCESS_KEY:-ignore}" \
  --secret-key "${S3_SECRET_KEY:-ignore}" \
  --endpoint-url "$ENDPOINT_URL" 


```markdown
This lists all the registries currently available in MLIS.
```

In [ ]:
!aioli r

## 2) Create the packaged model

This registers the model as a versioned packaged model (`bike-sharing-model`) backed by the registry + `MODEL_URL`.
Also sets CPU/memory limits to satisfy typical project quota policies (init/storage containers included).


In [ ]:
!aioli models create "$PACKAGED_MODEL" \
  --description "Bike sharing model from internal S3" \
  --image "$SERVE_IMAGE" \
  --url "$MODEL_URL" \
  --registry mlflow \
  --modelformat custom \
  --limits-cpu 2 \
  --limits-memory 2Gi


```markdown
This command lists all the packaged models currently available in MLIS. Use it to verify that your `bike-sharing-model` has been successfully registered.
```

In [ ]:
!aioli m

## 3) Create the deployment

Creates a secured endpoint (authentication required) and passes MLServer configuration via environment variables.


In [ ]:
!aioli deployments create "$DEPLOYMENT_NAME" \
  --model "$PACKAGED_MODEL" \
  --authentication-required true \
  --env MLSERVER_GRPC_PORT=9000 \
  --env MLSERVER_HTTP_PORT=8080 \
  --env MLSERVER_MODELS_DIR=/mnt/models \
  --env MLSERVER_MODEL_IMPLEMENTATION=mlserver_sklearn.SKLearnModel \
  --env MLSERVER_MODEL_NAME="$MODEL_NAME" \
  --env MLSERVER_MODEL_URI=/mnt/models


This command allows you to view deployments currently available in MLIS. Use this command to verify that your `bike-sharing-dep` deployment has been successfully created and is active.
```

In [ ]:
!aioli d

## 4) Inspect deployment state (including endpoint)

Use `aioli d show <deployment>` to see the `state.endpoint`.


In [ ]:
!aioli d show bike-sharing-dep

## 5) Example request

Example HTTP inference request against the deployed endpoint.
Adjust auth header/token and endpoint host for your environment.


In [ ]:
import os
import requests

DOMAIN_NAME = "svc.cluster.local" # change this to your domain for external access 
NAMESPACE = os.environ['NOTEBOOK_NAMESPACE']
DEPLOYMENT_NAME = os.environ['DEPLOYMENT_NAME']

if DOMAIN_NAME == "svc.cluster.local":
    PROTO = "http"
    SVC = f'{DEPLOYMENT_NAME}.{NAMESPACE}.{DOMAIN_NAME}'
    VERIFY = False
else:
    PROTO = "https"
    SVC = f'{DEPLOYMENT_NAME}.{NAMESPACE}.serving.{DOMAIN_NAME}'
    VERIFY = os.environ['EZUA_DOMAIN_CA_CERT_PATH']
URL = f"{PROTO}://{SVC}/v2/repository/index"

print(URL)


headers = {"Authorization": f"Bearer {os.environ['MLFLOW_TRACKING_TOKEN']}"}
response = requests.post(URL, json={}, verify=False, headers=headers)
response.json()

In [ ]:
MODEL_NAME = os.environ['MODEL_NAME']

URL = f"{PROTO}://{SVC}/v2/models/{MODEL_NAME}/infer"

names = ['season', 'year', 'month', 'hour_of_day', 'is_holiday', 'weekday', 'is_workingday', 
         'weather_situation', 'temperature', 'feels_like_temperature', 'humidity', 'windspeed']

input_data = [
    [1, 2, 1, 0, 0, 6, 0, 1, 0.24, 0.2879, 0.81, 0.0000],
    [1, 5, 1, 0, 0, 6, 1, 1, 0.24, 0.2879, 0.81, 0.0000]
]

inputs = {
  "inputs": [
    {
      "name": "ndarray",
      "shape": [2, 12],
      "datatype": "FP32",
      "data": input_data
    }
  ]
}

headers = {"Authorization": f"Bearer {os.environ['MLFLOW_TRACKING_TOKEN']}"}

response = requests.post(URL, json=inputs, headers=headers, verify=False)

print(response.reason)

output = response.json()['outputs'][0]['data']

print("Rented Bikes Per Hours:\n")
for item, out in zip(input_data, output):
    input_dict = dict(zip(names,item))
    print(f"Input Data: {input_dict} \n\nBike Per Hour: {out}\n")